# RLHF: Aligning GPT-2 for Hedged Financial Language

## Goal
Train GPT-2 using RLHF to prefer **factual, hedged financial language** over **confident/speculative language**.

**Example of what we want:**
- ✅ *"The stock may experience volatility given current market conditions"*
- ❌ *"The stock will definitely surge and reach new highs"*

## Pipeline Overview
```
Stage 1: SFT  →  Fine-tune GPT-2 on financial text prompts
Stage 2: RM   →  Train Reward Model on synthetic pairwise preferences  
Stage 3: PPO  →  RL fine-tuning using reward model + KL penalty
```

## Architecture
```
base_model (frozen SFT copy)  ←── KL penalty reference
      ↓
policy_model  →  generates completions
      ↓
reward_model  →  scores hedged vs speculative language
      ↓
PPO update  ←  reward - β * KL(policy || base_model)
```

## Setup & Installations

In [ ]:
# Install required libraries
# trl = Transformer Reinforcement Learning (HuggingFace)
!pip install trl transformers datasets accelerate torch -q

In [ ]:
import torch
import numpy as np
import random
import re
from typing import List, Tuple, Dict

from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    GPT2ForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from datasets import Dataset
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from trl.core import LengthSampler

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

---
## Stage 0: Define the Preference Signal

Before any training, we define **what we mean by hedged financial language**.
This is the human preference we're trying to encode into a reward model.

### Hedging vocabulary
We use two lexicons:
- **Hedging words**: epistemic uncertainty markers common in careful financial writing
- **Speculative words**: overconfident, promotional, or predictive language to penalize

In [ ]:
# ── Preference Lexicons ────────────────────────────────────────────────────────

HEDGE_WORDS = [
    # Epistemic hedges
    "may", "might", "could", "possibly", "potentially",
    "appears to", "seems to", "tends to", "suggests",
    # Uncertainty markers
    "uncertain", "unclear", "subject to", "risk", "risks",
    "volatility", "fluctuation", "variability",
    # Conditionality
    "if", "depending on", "assuming", "provided that",
    "contingent", "subject to change",
    # Analytical qualifiers  
    "historically", "based on", "according to", "indicates",
    "analysis suggests", "data shows", "evidence suggests"
]

SPECULATIVE_WORDS = [
    # Overconfident predictions
    "will definitely", "will certainly", "guaranteed", "guarantee",
    "without a doubt", "absolutely will", "100%", "sure to",
    # Hype language  
    "skyrocket", "explode", "moon", "to the moon", "unstoppable",
    "massive gains", "huge profits", "get rich", "can't lose",
    # Imperative speculation
    "you must buy", "buy now", "sell immediately", "don't miss",
    "once in a lifetime", "never been a better time",
    # Sensationalism
    "amazing", "incredible returns", "unbelievable", "shocking"
]

def compute_hedge_score(text: str) -> float:
    """
    Heuristic reward: measures the hedging quality of financial text.
    
    Returns a score in [0, 1] where:
      1.0 = well-hedged, cautious financial language
      0.0 = speculative, overconfident language
    
    This is our proxy for human preference — in a real RLHF setup,
    human annotators would provide pairwise labels instead.
    """
    text_lower = text.lower()
    words = text_lower.split()
    
    if len(words) == 0:
        return 0.5  # neutral for empty text
    
    # Count hedge signal
    hedge_count = sum(
        1 for h in HEDGE_WORDS if h in text_lower
    )
    
    # Count speculative signal  
    spec_count = sum(
        1 for s in SPECULATIVE_WORDS if s in text_lower
    )
    
    # Penalize ALL-CAPS (hype marker)
    caps_ratio = sum(1 for c in text if c.isupper()) / max(len(text), 1)
    caps_penalty = caps_ratio * 2
    
    # Penalize excessive exclamation marks
    exclaim_penalty = min(text.count('!') * 0.3, 1.0)
    
    # Raw score: hedge signal minus speculative signal
    raw = hedge_count - (spec_count * 1.5) - caps_penalty - exclaim_penalty
    
    # Sigmoid normalization to [0, 1]
    score = 1 / (1 + np.exp(-raw))
    
    return float(score)


# ── Sanity check the scoring function ────────────────────────────────────────
test_cases = [
    ("The stock may experience volatility depending on interest rate decisions.", "GOOD"),
    ("Based on historical data, earnings could potentially miss estimates.", "GOOD"),
    ("Analysis suggests the sector might face headwinds if inflation persists.", "GOOD"),
    ("This stock will DEFINITELY skyrocket! Guaranteed massive gains!", "BAD"),
    ("BUY NOW before it's too late! You can't lose! To the moon!!!", "BAD"),
    ("Get rich quick with this incredible investment opportunity!", "BAD"),
]

print("Hedge Scoring Sanity Check")
print("=" * 60)
for text, label in test_cases:
    score = compute_hedge_score(text)
    flag = "✅" if (label == "GOOD" and score > 0.5) or (label == "BAD" and score < 0.5) else "❌"
    print(f"{flag} [{label}] Score={score:.3f}")
    print(f"   {text[:70]}..." if len(text) > 70 else f"   {text}")
    print()

---
## Stage 1: Supervised Fine-Tuning (SFT)

We fine-tune GPT-2 on financial text prompts so it learns the domain vocabulary 
before RL begins. Without SFT, PPO would be trying to shape incoherent outputs.

**Note**: For a real project, you'd use a financial corpus (e.g., SEC filings, 
analyst reports). Here we use a small synthetic dataset to keep things runnable.

In [ ]:
# ── SFT Training Data ─────────────────────────────────────────────────────────
# Mix of hedged + speculative text so the base model knows both styles.
# The reward model will later discriminate between them.

SFT_DATA = [
    # Hedged examples (preferred)
    "The company's revenue may decline if macroeconomic conditions worsen.",
    "Based on current data, earnings could potentially miss analyst estimates.",
    "The stock appears to be undervalued relative to historical price-to-earnings ratios.",
    "Market volatility suggests investors should consider diversifying their portfolios.",
    "Analysis indicates the sector might face headwinds given rising interest rates.",
    "The quarterly results could reflect ongoing supply chain disruptions.",
    "Depending on Federal Reserve policy, the bond market may experience fluctuations.",
    "Historical trends suggest that inflation tends to peak before a rate cycle ends.",
    "The merger appears contingent on regulatory approval from antitrust authorities.",
    "Subject to market conditions, the IPO could be delayed until Q2.",
    "Earnings growth may slow if consumer spending contracts in the coming quarters.",
    "The portfolio's risk exposure could increase if equity correlations rise sharply.",
    "Evidence suggests the company might need to restructure its debt obligations.",
    "The dividend yield appears sustainable, assuming cash flows remain stable.",
    "Current valuations may not fully reflect the risks of a potential recession.",
    
    # Speculative examples (present in SFT for diversity, penalized in RL)
    "This stock will definitely hit $500 by end of year!",
    "Guaranteed returns await investors who buy now before prices explode.",
    "The market is going to absolutely skyrocket next quarter.",
    "You cannot lose money on this incredible investment opportunity.",
    "Massive gains are certain for early investors in this sector.",
    
    # Neutral financial domain text
    "The Federal Reserve announced its decision on interest rates yesterday.",
    "Q3 earnings reports are expected from major banks next week.",
    "The S&P 500 closed lower amid concerns about inflation data.",
    "Analysts revised their price targets following the earnings release.",
    "Trading volume increased significantly during the market open.",
]

FINANCIAL_PROMPTS = [
    "The stock market",
    "Investors should consider",
    "Based on recent earnings",
    "The company's outlook",
    "Market analysis suggests",
    "The Federal Reserve",
    "Portfolio risk",
    "Quarterly results",
    "The sector may",
    "Bond yields",
    "Equity valuation",
    "The merger could",
    "Consumer spending",
    "Inflation data",
    "The IPO",
]

print(f"SFT dataset: {len(SFT_DATA)} samples")
print(f"RL prompts pool: {len(FINANCIAL_PROMPTS)} prompts")

In [ ]:
# ── Load Tokenizer ─────────────────────────────────────────────────────────────
MODEL_NAME = "gpt2"  # 124M params — runs on CPU in ~minutes

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default
tokenizer.padding_side = "left"  # For generation tasks

print(f"Vocab size: {tokenizer.vocab_size}")
print(f"Model: {MODEL_NAME}")

In [ ]:
# ── Build SFT Dataset ─────────────────────────────────────────────────────────
def tokenize_sft(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    # For causal LM, labels = input_ids (predict next token)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

sft_dataset = Dataset.from_dict({"text": SFT_DATA})
sft_tokenized = sft_dataset.map(tokenize_sft, batched=True)
sft_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print(f"SFT tokenized dataset: {len(sft_tokenized)} samples")

In [ ]:
# ── SFT Training ──────────────────────────────────────────────────────────────
sft_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
sft_model.config.pad_token_id = tokenizer.eos_token_id

sft_args = TrainingArguments(
    output_dir="./sft_gpt2_finance",
    num_train_epochs=5,           # Small dataset, more epochs
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=10,
    save_strategy="no",
    report_to="none",             # Disable wandb/tensorboard
    fp16=torch.cuda.is_available(),
)

sft_trainer = Trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=sft_tokenized,
)

print("Starting SFT training...")
sft_trainer.train()
print("SFT complete ✅")

In [ ]:
# ── Inspect SFT Output ────────────────────────────────────────────────────────
def generate_text(model, prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

sft_model.to(device)
print("Sample outputs from SFT model (before RLHF):")
print("=" * 60)
for prompt in FINANCIAL_PROMPTS[:5]:
    text = generate_text(sft_model, prompt)
    score = compute_hedge_score(text)
    print(f"Prompt: '{prompt}'")
    print(f"Output: {text}")
    print(f"Hedge score: {score:.3f}")
    print()

---
## Stage 2: Reward Model Training

We train a **reward model** on synthetic pairwise preferences. This is the core 
of RLHF: instead of hand-crafting a reward function, we learn it from preference data.

### Pairwise preference setup
For each pair `(text_A, text_B)`, a human annotator says "I prefer A".  
We use our `compute_hedge_score` heuristic to simulate these annotations.

### Bradley-Terry loss
The reward model is trained with the **Bradley-Terry** preference model:

$$\mathcal{L}_{RM} = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma(r(x, y_w) - r(x, y_l)) \right]$$

Where $y_w$ is the preferred (winner) and $y_l$ is the dispreferred (loser) completion.
This directly encodes: *the preferred completion should get a higher reward score.*

In [ ]:
# ── Generate Pairwise Preference Dataset ─────────────────────────────────────
# Strategy: for each financial prompt, generate two completions using the SFT 
# model, then label the more hedged one as "preferred" using our heuristic.
# In real RLHF, human annotators would do this labeling step.

# Supplementary pairs: hand-crafted for better coverage
PAIRWISE_DATA = [
    # (preferred/hedged, dispreferred/speculative)
    (
        "The stock may experience significant volatility if interest rates rise further.",
        "The stock will definitely skyrocket once interest rates change!"
    ),
    (
        "Based on historical data, earnings could potentially disappoint investors.",
        "Earnings will absolutely blow past expectations and guarantee huge gains."
    ),
    (
        "The merger appears contingent on regulatory approval and may face delays.",
        "The merger will certainly go through and profits are guaranteed to explode."
    ),
    (
        "Inflation data suggests the central bank might consider pausing rate hikes.",
        "The Fed will 100% cut rates causing an unstoppable rally in equities!"
    ),
    (
        "Consumer spending could weaken if unemployment rises in coming quarters.",
        "Consumer spending will definitely surge — buy now before massive gains!"
    ),
    (
        "The company's guidance seems cautious, possibly reflecting supply risks.",
        "Guidance will shock everyone! Incredible returns are absolutely guaranteed!"
    ),
    (
        "Portfolio risk may increase if correlations between asset classes rise.",
        "Your portfolio will skyrocket! This is a once-in-a-lifetime opportunity!"
    ),
    (
        "The sector appears to be facing headwinds from rising input costs.",
        "The sector will EXPLODE higher! Massive gains are coming for everyone!"
    ),
    (
        "Bond yields could rise further depending on inflation trajectory.",
        "Bond yields will definitely collapse! Don't miss out on amazing returns!"
    ),
    (
        "Analyst estimates suggest Q3 revenue might fall short of consensus.",
        "Revenue will certainly beat every estimate! Get rich quick on this trade!"
    ),
    (
        "The IPO valuation seems stretched relative to comparable companies.",
        "The IPO will make you rich! 100% gains are absolutely coming your way!"
    ),
    (
        "Dividend sustainability appears uncertain given declining free cash flow.",
        "Dividends will skyrocket! You can't lose money on this amazing stock!"
    ),
    (
        "The acquisition could create value if integration risks are managed well.",
        "This acquisition will definitely make billions! Guaranteed unbelievable returns!"
    ),
    (
        "Currency risk may dampen overseas earnings if the dollar strengthens.",
        "Currency moves will certainly boost earnings! Profits are guaranteed to soar!"
    ),
    (
        "The credit rating could be downgraded if debt levels continue to rise.",
        "Credit will absolutely be upgraded! Amazing gains await every investor!"
    ),
]

print(f"Pairwise preference pairs: {len(PAIRWISE_DATA)}")

# Verify the heuristic agrees with our hand labels
correct = 0
for preferred, dispreferred in PAIRWISE_DATA:
    s1 = compute_hedge_score(preferred)
    s2 = compute_hedge_score(dispreferred)
    if s1 > s2:
        correct += 1
        
print(f"Heuristic agrees with hand labels: {correct}/{len(PAIRWISE_DATA)} ({100*correct/len(PAIRWISE_DATA):.0f}%)")

In [ ]:
# ── Reward Model Architecture ─────────────────────────────────────────────────
# We use GPT2ForSequenceClassification with num_labels=1 (scalar reward output).
# The final linear layer projects from hidden_size → 1, giving us r(x).

reward_model = GPT2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,  # Scalar reward
)
reward_model.config.pad_token_id = tokenizer.eos_token_id

# Initialize from SFT weights so reward model understands financial language
# Transfer all weights except the classification head
reward_model.transformer.load_state_dict(
    sft_model.transformer.state_dict()
)
reward_model = reward_model.to(device)
print("Reward model loaded with SFT backbone weights ✅")

In [ ]:
# ── Bradley-Terry Reward Model Trainer ───────────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader

class PreferenceDataset(torch.utils.data.Dataset):
    """Dataset of pairwise preferences for reward model training."""
    
    def __init__(self, pairs: List[Tuple[str, str]], tokenizer, max_length: int = 128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.pairs)
    
    def __getitem__(self, idx):
        preferred, dispreferred = self.pairs[idx]
        
        # Tokenize both completions
        enc_w = self.tokenizer(
            preferred, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        enc_l = self.tokenizer(
            dispreferred, truncation=True, max_length=self.max_length,
            padding="max_length", return_tensors="pt"
        )
        
        return {
            "input_ids_w": enc_w["input_ids"].squeeze(),
            "attention_mask_w": enc_w["attention_mask"].squeeze(),
            "input_ids_l": enc_l["input_ids"].squeeze(),
            "attention_mask_l": enc_l["attention_mask"].squeeze(),
        }


def bradley_terry_loss(reward_w: torch.Tensor, reward_l: torch.Tensor) -> torch.Tensor:
    """
    Bradley-Terry preference model loss.
    
    L = -log(sigmoid(r_w - r_l))
    
    Minimizing this pushes r_w > r_l, making the reward model
    assign higher scores to preferred (hedged) completions.
    """
    return -torch.log(torch.sigmoid(reward_w - reward_l)).mean()


def train_reward_model(
    model: nn.Module,
    pairs: List[Tuple[str, str]],
    tokenizer,
    num_epochs: int = 10,
    lr: float = 1e-5,
    batch_size: int = 4,
) -> List[float]:
    """Train reward model with Bradley-Terry loss."""
    
    dataset = PreferenceDataset(pairs, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    losses = []
    model.train()
    
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch in loader:
            # Get rewards for preferred and dispreferred completions
            r_w = model(
                input_ids=batch["input_ids_w"].to(device),
                attention_mask=batch["attention_mask_w"].to(device)
            ).logits.squeeze(-1)  # shape: (batch,)
            
            r_l = model(
                input_ids=batch["input_ids_l"].to(device),
                attention_mask=batch["attention_mask_l"].to(device)
            ).logits.squeeze(-1)  # shape: (batch,)
            
            loss = bradley_terry_loss(r_w, r_l)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        losses.append(avg_loss)
        
        if (epoch + 1) % 2 == 0:
            print(f"  Epoch {epoch+1:2d}/{num_epochs} | BT Loss: {avg_loss:.4f}")
    
    return losses


print("Training reward model with Bradley-Terry loss...")
rm_losses = train_reward_model(reward_model, PAIRWISE_DATA, tokenizer, num_epochs=10)
print("\nReward model training complete ✅")

In [ ]:
# ── Evaluate Reward Model ─────────────────────────────────────────────────────
def get_reward(model, text: str) -> float:
    """Get scalar reward from reward model for a given text."""
    model.eval()
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=128,
        padding="max_length"
    ).to(device)
    
    with torch.no_grad():
        reward = model(**inputs).logits.squeeze().item()
    return reward


print("Reward Model Evaluation")
print("=" * 65)
print(f"{'Text (truncated)':<45} {'RM Score':>8} {'Heuristic':>9}")
print("-" * 65)

eval_texts = [
    "The stock may decline if economic conditions worsen.",
    "Based on data, earnings could potentially miss estimates.",
    "This stock will definitely make you rich with guaranteed gains!",
    "BUY NOW! Skyrocket to the moon! 100% profit guaranteed!",
    "The company appears to face headwinds from rising costs.",
    "Massive unstoppable gains are absolutely certain this year!",
]

for text in eval_texts:
    rm_score = get_reward(reward_model, text)
    heuristic = compute_hedge_score(text)
    truncated = text[:43] + ".." if len(text) > 45 else text
    print(f"{truncated:<45} {rm_score:>8.3f} {heuristic:>9.3f}")

---
## Stage 3: RL Fine-Tuning with PPO

We now use **Proximal Policy Optimization (PPO)** to fine-tune the SFT model,
guided by the reward model. The KL divergence penalty prevents the policy 
from drifting too far from the SFT reference model.

### PPO Objective
$$\mathcal{L}_{PPO} = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$

### Total Reward with KL Penalty
$$R(x, y) = r_{\theta_{RM}}(x, y) - \beta \cdot \text{KL}\left(\pi_{\theta}(y|x) \| \pi_{\text{SFT}}(y|x)\right)$$

The $\beta$ coefficient controls the tradeoff:
- **High β**: Stay close to SFT, less reward hacking, but slower improvement
- **Low β**: Maximize reward more aggressively, but risk reward hacking

In [ ]:
# ── PPO Configuration ─────────────────────────────────────────────────────────
ppo_config = PPOConfig(
    model_name=MODEL_NAME,
    learning_rate=1e-5,
    batch_size=8,            # Number of prompts per PPO step
    mini_batch_size=4,       # Mini-batch for gradient updates
    gradient_accumulation_steps=1,
    ppo_epochs=4,            # Inner PPO epochs per batch
    kl_penalty="kl",         # Use standard KL divergence
    init_kl_coef=0.2,        # β — KL penalty coefficient
    target_kl=6.0,           # Adaptive KL target (auto-adjusts β)
    adap_kl_ctrl=True,       # Enable adaptive KL control
    cliprange=0.2,           # PPO clip ratio ε
    vf_coef=0.1,             # Value function loss coefficient
    seed=42,
    log_with=None,           # Disable wandb
)

print("PPO Config:")
print(f"  KL penalty β: {ppo_config.init_kl_coef} (adaptive target: {ppo_config.target_kl})")
print(f"  PPO clip ε: {ppo_config.cliprange}")
print(f"  Batch size: {ppo_config.batch_size}")

In [ ]:
# ── Load Policy and Reference Models ─────────────────────────────────────────
# AutoModelForCausalLMWithValueHead adds a value head (V(s)) on top of GPT-2,
# which PPO needs to estimate advantage A(s,a) = Q(s,a) - V(s).

# Policy model: initialized from SFT weights, will be updated by PPO
policy_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME)

# Load SFT weights into policy model's transformer
policy_model.pretrained_model.transformer.load_state_dict(
    sft_model.transformer.state_dict()
)
policy_model.pretrained_model.lm_head.load_state_dict(
    sft_model.lm_head.state_dict()
)

# Reference model: frozen SFT copy used for KL penalty
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(MODEL_NAME)
ref_model.pretrained_model.transformer.load_state_dict(
    sft_model.transformer.state_dict()
)
ref_model.pretrained_model.lm_head.load_state_dict(
    sft_model.lm_head.state_dict()
)

# Freeze reference model — it must NOT be updated during PPO
for param in ref_model.parameters():
    param.requires_grad = False

print("Policy model (will be updated): ✅")
print("Reference model (frozen SFT):   ✅")

In [ ]:
# ── Build PPO Prompt Dataset ───────────────────────────────────────────────────
# PPO needs tokenized prompts. The policy generates completions from these,
# then the reward model scores them.

# Expand prompt pool with variations
EXPANDED_PROMPTS = FINANCIAL_PROMPTS + [
    "The company's financial",
    "Recent economic data",
    "Investment risk",
    "The market outlook",
    "Analysts believe",
    "The central bank",
    "Corporate earnings",
    "The fund's performance",
]

def build_ppo_dataset(prompts: List[str], tokenizer) -> Dataset:
    """Tokenize prompts for PPO training."""
    tokenized = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=32,  # Short prompts — model generates the rest
        return_tensors="pt"
    )
    dataset = Dataset.from_dict({
        "input_ids": tokenized["input_ids"].tolist(),
        "query": prompts,
    })
    return dataset

ppo_dataset = build_ppo_dataset(EXPANDED_PROMPTS, tokenizer)
print(f"PPO prompt dataset: {len(ppo_dataset)} prompts")

In [ ]:
# ── PPO Trainer ───────────────────────────────────────────────────────────────
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
    data_collator=lambda data: {
        "input_ids": [torch.tensor(d["input_ids"]) for d in data],
        "query": [d["query"] for d in data],
    },
)

# Generation config: how the policy generates completions during PPO
generation_kwargs = {
    "min_length": 20,
    "max_new_tokens": 50,
    "top_k": 50,
    "top_p": 0.9,
    "do_sample": True,
    "temperature": 0.8,
    "pad_token_id": tokenizer.eos_token_id,
}

print("PPO Trainer initialized ✅")

In [ ]:
# ── PPO Training Loop ─────────────────────────────────────────────────────────
# This is the core RLHF loop:
#
# For each batch:
#   1. Policy generates completions from prompts
#   2. Reward model scores each completion
#   3. PPO updates policy to increase expected reward
#      while staying close to the reference (KL penalty)

NUM_PPO_STEPS = 30  # Increase for better results (try 100-200 for real training)

training_log = []

print(f"Starting PPO training for {NUM_PPO_STEPS} steps...")
print("=" * 65)

for step, batch in enumerate(ppo_trainer.dataloader):
    if step >= NUM_PPO_STEPS:
        break
    
    query_tensors = batch["input_ids"]
    
    # ── Step 1: Generate completions from policy ──────────────────────────
    response_tensors = ppo_trainer.generate(
        query_tensors,
        return_prompt=False,
        **generation_kwargs
    )
    
    # Decode completions (just the generated part, not the prompt)
    batch_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
    
    # ── Step 2: Score completions with reward model ───────────────────────
    rewards = []
    for text in batch_texts:
        # Learned reward from reward model
        rm_reward = get_reward(reward_model, text)
        rewards.append(torch.tensor(rm_reward, dtype=torch.float32))
    
    # ── Step 3: PPO update ────────────────────────────────────────────────
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
    
    # Log metrics
    mean_reward = np.mean([r.item() for r in rewards])
    mean_kl = stats.get("objective/kl", 0)
    mean_hedge = np.mean([compute_hedge_score(t) for t in batch_texts])
    
    training_log.append({
        "step": step,
        "mean_reward": mean_reward,
        "kl_divergence": mean_kl,
        "hedge_score": mean_hedge,
    })
    
    if (step + 1) % 5 == 0:
        print(f"Step {step+1:3d} | Reward: {mean_reward:+.3f} | KL: {mean_kl:.3f} | Hedge: {mean_hedge:.3f}")

print("\nPPO training complete ✅")

---
## Evaluation: Before vs After RLHF

In [ ]:
# ── Side-by-side comparison: SFT vs RLHF ────────────────────────────────────
import textwrap

def generate_rlhf(model, prompt: str, max_new_tokens: int = 60) -> str:
    """Generate text from the PPO-trained policy model."""
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    model.pretrained_model.eval()
    with torch.no_grad():
        output = model.pretrained_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


eval_prompts = [
    "The stock market",
    "Investors should consider",
    "The company's outlook",
    "Market analysis suggests",
    "The Federal Reserve",
]

print("RLHF EVALUATION: SFT vs PPO-trained Policy")
print("=" * 70)

results = []
for prompt in eval_prompts:
    sft_text = generate_text(sft_model, prompt)
    rlhf_text = generate_rlhf(policy_model, prompt)
    
    sft_score = compute_hedge_score(sft_text)
    rlhf_score = compute_hedge_score(rlhf_text)
    improvement = rlhf_score - sft_score
    
    results.append({
        "prompt": prompt,
        "sft_score": sft_score,
        "rlhf_score": rlhf_score,
        "improvement": improvement,
    })
    
    print(f"\nPrompt: '{prompt}'")
    print(f"  SFT  [{sft_score:.3f}]: {textwrap.shorten(sft_text, 80)}")
    print(f"  RLHF [{rlhf_score:.3f}]: {textwrap.shorten(rlhf_text, 80)}")
    delta_str = f"+{improvement:.3f}" if improvement > 0 else f"{improvement:.3f}"
    flag = "✅" if improvement > 0 else "⚠️"
    print(f"  Change: {delta_str} {flag}")

print("\n" + "=" * 70)
avg_improvement = np.mean([r["improvement"] for r in results])
print(f"Average hedge score improvement: {avg_improvement:+.3f}")

In [ ]:
# ── Plot Training Curves ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

if training_log:
    steps = [d["step"] for d in training_log]
    rewards = [d["mean_reward"] for d in training_log]
    kls = [d["kl_divergence"] for d in training_log]
    hedges = [d["hedge_score"] for d in training_log]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("RLHF Training: Aligning GPT-2 for Hedged Financial Language", 
                 fontsize=13, fontweight="bold")
    
    # Mean Reward
    axes[0].plot(steps, rewards, color="#2196F3", linewidth=2)
    axes[0].set_title("Mean Reward (RM Score)")
    axes[0].set_xlabel("PPO Step")
    axes[0].set_ylabel("Reward")
    axes[0].axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    axes[0].grid(True, alpha=0.3)
    
    # KL Divergence
    axes[1].plot(steps, kls, color="#FF5722", linewidth=2)
    axes[1].set_title("KL Divergence (policy vs SFT)")
    axes[1].set_xlabel("PPO Step")
    axes[1].set_ylabel("KL")
    axes[1].grid(True, alpha=0.3)
    
    # Hedge Score
    axes[2].plot(steps, hedges, color="#4CAF50", linewidth=2)
    axes[2].set_title("Hedge Score (heuristic)")
    axes[2].set_xlabel("PPO Step")
    axes[2].set_ylabel("Score [0,1]")
    axes[2].set_ylim(0, 1)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("rlhf_training_curves.png", dpi=120, bbox_inches="tight")
    plt.show()
    print("Training curves saved ✅")

---
## Understanding Reward Hacking

A critical concept in RLHF. Watch what happens when we run PPO **without** the KL penalty.

In [ ]:
# ── Reward Hacking Demo ───────────────────────────────────────────────────────
# To demonstrate reward hacking, let's see what the model might "learn"
# if it over-optimizes the reward without KL constraint.

# Construct examples that game our heuristic reward without being genuinely hedged
reward_hacked_examples = [
    # Stuffing hedge words with no coherent meaning
    "May might could possibly potentially uncertain risk volatility if depending on contingent subject to change.",
    # Hedge words but completely incoherent
    "The stock may could possibly might uncertain possibly may could depending on if.",
    # Genuine hedged text (what we actually want)
    "The stock may decline if inflation persists beyond the current quarter.",
    # Coherent but no hedging
    "The stock rose sharply after earnings were released yesterday.",
]

print("Reward Hacking Illustration")
print("=" * 65)
print("Shows why KL penalty matters — heuristic can be gamed")
print()
print(f"{'Type':<20} {'Heuristic':>9} {'Text (truncated)'}")
print("-" * 65)
labels = ["HACKED (gamed)", "HACKED (gamed)", "GENUINE hedge", "No hedge"]
for text, label in zip(reward_hacked_examples, labels):
    score = compute_hedge_score(text)
    truncated = text[:35] + "..." if len(text) > 35 else text
    print(f"{label:<20} {score:>9.3f} {truncated}")

print()
print("⚠️  The KL penalty prevents the model from drifting toward")
print("    incoherent hedge-word stuffing by anchoring it to SFT.")
print()
print("   R_total = r_RM(text) - β × KL(policy || SFT)")
print(f"   With β = {ppo_config.init_kl_coef}, incoherent text is heavily penalized.")

---
## Summary & Key Takeaways

### What we built
| Stage | Component | Purpose |
|-------|-----------|----------|
| SFT | GPT-2 fine-tuned on financial text | Teaches domain vocabulary |
| RM | GPT-2 classification head | Encodes human preference for hedged language |
| PPO | Policy + value head + KL penalty | RL optimization with stability |

### Critical concepts demonstrated

**1. Bradley-Terry loss** — trains the reward model from pairwise comparisons, not direct ratings. This is the standard RLHF approach because pairwise judgments are more reliable for annotators than absolute scores.

**2. KL divergence penalty** — the most important stabilizer in RLHF. Without it, the policy collapses into reward hacking (hedge-word stuffing) within a few PPO steps.

**3. Adaptive KL controller** — automatically adjusts β to keep KL near a target value, preventing both under-optimization (too conservative) and over-optimization (reward hacking).

**4. Value head** — PPO needs V(s) to compute advantage estimates A = Q - V. This is why `AutoModelForCausalLMWithValueHead` is separate from the plain LM.

### Extensions to try
- Replace heuristic reward with a trained classifier on real financial corpora
- Use `gpt2-medium` for better generation quality
- Add a **harmlessness** reward component (penalize misleading claims)
- Experiment with different β values and observe reward hacking onset
- Try **DPO** (Direct Preference Optimization) as an RL-free alternative